# GRPO Chess Colab Runner

Run the full searchless pipeline in Colab:
1. Pretrain
2. Distill (warm-start from pretrain)
3. GRPO (warm-start from distill)

This notebook generates either smoke-test or full-run configs (`RUN_PROFILE`) for end-to-end execution.


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "feature/lightning_training"  #@param {type:"string"}
RUN_PROFILE = "full"  #@param ["smoke", "full"]
USE_DRIVE = True  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/data/grpo-chess"  #@param {type:"string"}
HF_CACHE_ROOT = "/content/drive/MyDrive/data/grpo-chess/hf_cache"  #@param {type:"string"}
USE_WANDB = True  #@param {type:"boolean"}
WANDB_API_KEY = ""  #@param {type:"string"}
WANDB_PROJECT_PRETRAIN = "chess-grpo-pretrain"  #@param {type:"string"}
WANDB_PROJECT_DISTILL = "chess-grpo-pretrain"  #@param {type:"string"}
WANDB_PROJECT_GRPO = "Chess-GRPO-Bot"  #@param {type:"string"}
RUN_PRETRAIN = True  #@param {type:"boolean"}
RUN_DISTILL = True  #@param {type:"boolean"}
RUN_GRPO = True  #@param {type:"boolean"}


In [ ]:
# Setup workspace
import os
import sys
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

hf_cache_root = Path(HF_CACHE_ROOT if USE_DRIVE else '/content/hf_cache')
hf_cache_root.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(hf_cache_root)
os.environ['HF_DATASETS_CACHE'] = str(hf_cache_root / 'datasets')
os.environ['TRANSFORMERS_CACHE'] = str(hf_cache_root / 'transformers')
print(f"HF cache root: {hf_cache_root}")

if USE_WANDB:
    wandb_key = WANDB_API_KEY.strip()
    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = (userdata.get('WANDB_API_KEY') or userdata.get('WANDB_KEY') or '').strip()
        except Exception:
            wandb_key = ''

    if not wandb_key:
        raise RuntimeError(
            'USE_WANDB=True but no key was found. Set WANDB_API_KEY in runtime params or Colab Secrets.'
        )

    os.environ['WANDB_API_KEY'] = wandb_key
    os.environ['WANDB_KEY'] = wandb_key
    print('WandB key configured via environment; logger login happens during training.')
else:
    os.environ['WANDB_DISABLED'] = 'true'
    print('WandB disabled for this notebook run.')

!rm -rf /content/grpo_chess
!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git fetch --all --tags
!git checkout {REPO_REF}
!git submodule update --init --recursive

if '/content/grpo_chess' not in sys.path:
    sys.path.append('/content/grpo_chess')


In [ ]:
# Install dependencies
%pip install -q --upgrade pip
%pip install -q -r requirements.txt
%pip install -q pytorch_lightning wandb python-chess jaxtyping datasets huggingface_hub
!apt-get -qq update
!apt-get -qq install -y stockfish
!which stockfish || true
!python -V


## Create Pipeline Configs

These configs are generated inside `src/configs/` based on `RUN_PROFILE`.
- `smoke`: fast wiring check
- `full`: full-size training profile aligned with repo defaults


In [ ]:
from pathlib import Path

repo = Path('/content/grpo_chess')
config_dir = repo / 'src' / 'configs'
config_dir.mkdir(parents=True, exist_ok=True)

run_profile = RUN_PROFILE.strip().lower()
if run_profile not in {'smoke', 'full'}:
    raise ValueError(f'RUN_PROFILE must be one of ["smoke", "full"], got: {RUN_PROFILE}')

profile_suffix = f'colab_e2e_{run_profile}'
artifact_root = Path(DRIVE_ROOT) / profile_suffix if USE_DRIVE else Path(f'/content/{profile_suffix}')
pretrain_ckpt_dir = artifact_root / 'checkpoints' / 'pretrain'
distill_ckpt_dir = artifact_root / 'checkpoints' / 'distill'
grpo_ckpt_dir = artifact_root / 'checkpoints' / 'grpo'
distill_data_dir = artifact_root / 'distill_data'
pretrain_processed_cache_dir = hf_cache_root / 'pretrain_processed'
hf_datasets_cache_dir = hf_cache_root / 'datasets'

for d in [
    pretrain_ckpt_dir,
    distill_ckpt_dir,
    grpo_ckpt_dir,
    distill_data_dir,
    pretrain_processed_cache_dir,
    hf_datasets_cache_dir,
]:
    d.mkdir(parents=True, exist_ok=True)

pretrain_cfg_name = f'pretrain_colab_e2e_{run_profile}.yaml'
distill_cfg_name = f'distill_colab_e2e_{run_profile}.yaml'
grpo_cfg_name = f'grpo_colab_e2e_{run_profile}.yaml'

wandb_flag = 'true' if USE_WANDB else 'false'
distill_auto_disable_wandb = 'false' if USE_WANDB else 'true'

if run_profile == 'full':
    profile = {
        'pretrain_batch_size': 4096,
        'pretrain_num_epochs': 22,
        'pretrain_warmup_steps': 1000,
        'pretrain_num_workers': 4,
        'pretrain_val_check_interval': 0.1,
        'pretrain_eval_every_n_epochs': 1,
        'pretrain_eval_games': 32,
        'pretrain_eval_max_plies': 400,
        'pretrain_opening_plies': 6,
        'pretrain_stockfish_movetime_ms': 50,
        'pretrain_max_samples': 5000000,
        'pretrain_sample_positions_per_game': 3,
        'pretrain_buffer_size': 10000,
        'distill_teacher_model': '136M',
        'distill_teacher_batch_size': 512,
        'distill_generate_shard_size': 50000,
        'distill_generate_max_samples': 2000000,
        'distill_generate_sample_positions_per_game': 3,
        'deepmind_num_shards': 50,
        'distill_batch_size': 2048,
        'distill_num_epochs': 10,
        'distill_warmup_steps': 1000,
        'distill_num_workers': 4,
        'distill_val_check_interval': 0.1,
        'distill_eval_every_n_epochs': 1,
        'distill_eval_games': 32,
        'distill_eval_max_plies': 400,
        'distill_opening_plies': 6,
        'distill_stockfish_movetime_ms': 50,
        'distill_max_shards': 'null',
        'grpo_num_epochs': 400,
        'grpo_batch_size': 32,
        'grpo_steps_per_epoch': 512,
        'grpo_checkpoint_every_n_epochs': 5,
        'grpo_keep_n_checkpoints': 3,
        'grpo_num_trajectories': 16,
        'grpo_trajectory_depth': 16,
        'grpo_eval_every_n_epochs': 10,
        'grpo_rollout_temperature': 1.3,
        'grpo_teacher_forcing_prob': 0.1,
        'grpo_eval_games': 64,
        'grpo_eval_max_plies': 400,
        'grpo_opening_plies': 6,
        'grpo_stockfish_movetime_ms': 50,
        'grpo_hash_mb': 128,
        'grpo_freeze_layers': 2,
    }
else:
    profile = {
        'pretrain_batch_size': 512,
        'pretrain_num_epochs': 1,
        'pretrain_warmup_steps': 100,
        'pretrain_num_workers': 2,
        'pretrain_val_check_interval': 1.0,
        'pretrain_eval_every_n_epochs': 1000,
        'pretrain_eval_games': 4,
        'pretrain_eval_max_plies': 150,
        'pretrain_opening_plies': 4,
        'pretrain_stockfish_movetime_ms': 20,
        'pretrain_max_samples': 20000,
        'pretrain_sample_positions_per_game': 1,
        'pretrain_buffer_size': 5000,
        'distill_teacher_model': '9M',
        'distill_teacher_batch_size': 256,
        'distill_generate_shard_size': 20000,
        'distill_generate_max_samples': 50000,
        'distill_generate_sample_positions_per_game': 2,
        'deepmind_num_shards': 1,
        'distill_batch_size': 512,
        'distill_num_epochs': 1,
        'distill_warmup_steps': 100,
        'distill_num_workers': 2,
        'distill_val_check_interval': 1.0,
        'distill_eval_every_n_epochs': 1000,
        'distill_eval_games': 4,
        'distill_eval_max_plies': 150,
        'distill_opening_plies': 4,
        'distill_stockfish_movetime_ms': 20,
        'distill_max_shards': 1,
        'grpo_num_epochs': 1,
        'grpo_batch_size': 8,
        'grpo_steps_per_epoch': 32,
        'grpo_checkpoint_every_n_epochs': 1,
        'grpo_keep_n_checkpoints': 1,
        'grpo_num_trajectories': 4,
        'grpo_trajectory_depth': 6,
        'grpo_eval_every_n_epochs': 1000,
        'grpo_rollout_temperature': 1.2,
        'grpo_teacher_forcing_prob': 0.0,
        'grpo_eval_games': 4,
        'grpo_eval_max_plies': 150,
        'grpo_opening_plies': 4,
        'grpo_stockfish_movetime_ms': 20,
        'grpo_hash_mb': 64,
        'grpo_freeze_layers': 1,
    }

pretrain_cfg = f"""pretrain:
  lr: 0.0001
  batch_size: {profile['pretrain_batch_size']}
  num_epochs: {profile['pretrain_num_epochs']}
  warmup_steps: {profile['pretrain_warmup_steps']}
  weight_decay: 0.01
  max_grad_norm: 1.0
  checkpoint_dir: "{pretrain_ckpt_dir}"
  resume_from: null
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_PRETRAIN}"
  label_smoothing: 0.1
  num_workers: {profile['pretrain_num_workers']}
  val_check_interval: {profile['pretrain_val_check_interval']}
  eval_every_n_epochs: {profile['pretrain_eval_every_n_epochs']}

eval:
  games: {profile['pretrain_eval_games']}
  seed: 0
  max_plies: {profile['pretrain_eval_max_plies']}
  randomize_opening: true
  opening_plies: {profile['pretrain_opening_plies']}

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  movetime_ms: {profile['pretrain_stockfish_movetime_ms']}

policy:
  greedy: true

dataset:
  min_elo: 1800
  max_samples: {profile['pretrain_max_samples']}
  skip_first_n_moves: 5
  skip_last_n_moves: 5
  sample_positions_per_game: {profile['pretrain_sample_positions_per_game']}
  buffer_size: {profile['pretrain_buffer_size']}
  filter_abandoned: true
  dataset_name: "Lichess/standard-chess-games"
  split: "train"
  is_eval: false
  eval_fraction: 0.05
  cache_path: "{pretrain_processed_cache_dir}"
  hf_cache_dir: "{hf_datasets_cache_dir}"

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968
"""

distill_cfg = f"""generate:
  teacher_model: "{profile['distill_teacher_model']}"
  checkpoint_dir: "searchless_chess/checkpoints"
  checkpoint_step: 6400000
  teacher_batch_size: {profile['distill_teacher_batch_size']}
  top_k: 8
  teacher_temperature: 1.0
  output_dir: "{distill_data_dir}"
  shard_size: {profile['distill_generate_shard_size']}
  min_elo: 1800
  max_samples: {profile['distill_generate_max_samples']}
  skip_first_n_moves: 5
  skip_last_n_moves: 5
  sample_positions_per_game: {profile['distill_generate_sample_positions_per_game']}

deepmind_data:
  num_shards: {profile['deepmind_num_shards']}
  top_k: 8
  temperature: 1.0
  min_win_prob: 0.55
  output_dir: "{distill_data_dir}"
  shard_size: {profile['distill_generate_shard_size']}

distill:
  lr: 0.0001
  batch_size: {profile['distill_batch_size']}
  num_epochs: {profile['distill_num_epochs']}
  warmup_steps: {profile['distill_warmup_steps']}
  weight_decay: 0.01
  max_grad_norm: 1.0
  checkpoint_dir: "{distill_ckpt_dir}"
  pretrain_checkpoint: "{pretrain_ckpt_dir / 'pretrain_final.pt'}"
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_DISTILL}"
  log_model_artifacts: false
  auto_disable_wandb_if_missing_key: {distill_auto_disable_wandb}
  fail_on_nonfinite: false
  max_nonfinite_batches: 50
  num_workers: {profile['distill_num_workers']}
  val_check_interval: {profile['distill_val_check_interval']}
  eval_every_n_epochs: {profile['distill_eval_every_n_epochs']}

eval:
  games: {profile['distill_eval_games']}
  seed: 0
  max_plies: {profile['distill_eval_max_plies']}
  randomize_opening: true
  opening_plies: {profile['distill_opening_plies']}

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  movetime_ms: {profile['distill_stockfish_movetime_ms']}

policy:
  greedy: true

dataset:
  data_dir: "{distill_data_dir}"
  top_k: 8
  eval_fraction: 0.05
  max_shards: {profile['distill_max_shards']}

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968
"""

grpo_cfg = f"""training:
  num_epochs: {profile['grpo_num_epochs']}
  batch_size: {profile['grpo_batch_size']}
  steps_per_epoch: {profile['grpo_steps_per_epoch']}
  checkpoint_dir: "{grpo_ckpt_dir}"
  checkpoint_every_n_epochs: {profile['grpo_checkpoint_every_n_epochs']}
  keep_n_checkpoints: {profile['grpo_keep_n_checkpoints']}
  use_wandb: {wandb_flag}
  wandb_project: "{WANDB_PROJECT_GRPO}"

grpo:
  lr: 0.000001
  num_trajectories: {profile['grpo_num_trajectories']}
  trajectory_depth: {profile['grpo_trajectory_depth']}
  clip_ratio: 0.20
  kl_coef: 0.001
  eval_every_n_epochs: {profile['grpo_eval_every_n_epochs']}
  ppo_steps: 1
  rollout_temperature: {profile['grpo_rollout_temperature']}
  enable_safety_checks: false
  safety_patience_steps: 1000
  max_clip_fraction: 0.95
  teacher_forcing_prob: {profile['grpo_teacher_forcing_prob']}
  teacher_forcing_depth: 4

transformer:
  vocab_size: 300
  embed_dim: 256
  num_layers: 4
  num_heads: 8
  action_dim: 1968

eval:
  games: {profile['grpo_eval_games']}
  seed: 0
  max_plies: {profile['grpo_eval_max_plies']}
  randomize_opening: true
  opening_plies: {profile['grpo_opening_plies']}

stockfish:
  path: "/usr/games/stockfish"
  skill_level: 2
  use_elo_limit: false
  elo: 2500
  movetime_ms: {profile['grpo_stockfish_movetime_ms']}
  threads: 1
  hash_mb: {profile['grpo_hash_mb']}

policy:
  temperature: 0.8
  greedy: true
  branching_factor: 4
  search_depth: 2

searcher: null

pretrain:
  checkpoint_path: "{distill_ckpt_dir / 'distill_final.pt'}"
  freeze_layers: {profile['grpo_freeze_layers']}

dataset:
  max_steps: {profile['grpo_steps_per_epoch']}
  phase_distribution:
    opening: 0.33
    middlegame: 0.34
    endgame: 0.33
  min_eval_cp: -200
  max_eval_cp: 200
  quality_filter: true
  stockfish_filter_depth: 4
"""

(config_dir / pretrain_cfg_name).write_text(pretrain_cfg)
(config_dir / distill_cfg_name).write_text(distill_cfg)
(config_dir / grpo_cfg_name).write_text(grpo_cfg)

print('Run profile:', run_profile)
print('Wrote configs:')
print('-', pretrain_cfg_name)
print('-', distill_cfg_name)
print('-', grpo_cfg_name)
print('Artifact root:', artifact_root)
print('HF datasets cache:', hf_datasets_cache_dir)


## Run Pipeline Stages


In [ ]:
# Stage 1: Pretrain
if RUN_PRETRAIN:
    !python -m src.pretrain.pretrain --config {pretrain_cfg_name}
else:
    print('Skipping pretrain stage')


In [ ]:
# Stage 2: Distill (warm-start from pretrain_final.pt)
if RUN_DISTILL:
    !python -m src.distill.distill --config {distill_cfg_name}
else:
    print('Skipping distill stage')


In [ ]:
# Stage 3: GRPO (warm-start from distill_final.pt)
if RUN_GRPO:
    !python -m src.train_self_play --config {grpo_cfg_name}
else:
    print('Skipping GRPO stage')


In [ ]:
# Verify expected artifacts
from pathlib import Path

checks = {
    'pretrain_final': pretrain_ckpt_dir / 'pretrain_final.pt',
    'distill_final': distill_ckpt_dir / 'distill_final.pt',
    'grpo_dir': grpo_ckpt_dir,
}
for name, path in checks.items():
    print(f'{name}:', 'OK' if path.exists() else f'MISSING ({path})')


## Notes

- Set `RUN_PROFILE="full"` for full-size configs, or `RUN_PROFILE="smoke"` for fast wiring checks.
- This project is searchless; this notebook does not introduce tree-search/MCTS steps.
